<a href="https://colab.research.google.com/github/HS-base/FYP/blob/main/Blend_Properties_Calculation_for_CFD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install CoolProp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 37.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import CoolProp.CoolProp as CP

def fill_missing_data(data_array):
    """Linearly interpolates missing (NaN) values to patch CoolProp failure gaps."""
    nans, x = np.isnan(data_array), lambda z: z.nonzero()[0]
    data_array[nans] = np.interp(x(nans), x(~nans), data_array[~nans])
    return data_array

def generate_lookup_tables():
    fluid_string = "Propane&Ammonia"
    AS = CP.AbstractState("HEOS", fluid_string)
    pressures_kPa = [971.5, 1300, 3000]

    # Use exactly 101 points for highly efficient array indexing in C
    N_points = 101
    y_propane_array = np.linspace(0, 1.0, N_points)

    print("/* =========================================================================")
    print("   ANSYS Fluent UDF 1D Lookup Tables: Propane & Ammonia")
    print("   Y_propane resolution: 0.01 (101 points total)")
    print("========================================================================== */\n")

    for P_kPa in pressures_kPa:
        P_Pa = P_kPa * 1000.0

        T_bubble = np.zeros(N_points)
        Hfg_mix = np.zeros(N_points)
        Cp_mix_liq = np.zeros(N_points)
        Cp_mix_vap = np.zeros(N_points)

        # Calculate properties for each mass fraction
        for idx, y0 in enumerate(y_propane_array):
            y1 = 1.0 - y0  # Ammonia MASS fraction
            AS.set_mass_fractions([y0, y1])

            try:
                AS.update(CP.PQ_INPUTS, P_Pa, 0.0)
                T_bubble[idx] = AS.T()
                Cp_mix_liq[idx] = AS.cpmass()
                h_l = AS.hmass()

                AS.update(CP.PQ_INPUTS, P_Pa, 1.0)
                Cp_mix_vap[idx] = AS.cpmass()
                h_v = AS.hmass()

                Hfg_mix[idx] = h_v - h_l

            except ValueError:
                # If CoolProp fails, assign NaN temporarily
                T_bubble[idx] = np.nan
                Hfg_mix[idx] = np.nan
                Cp_mix_liq[idx] = np.nan
                Cp_mix_vap[idx] = np.nan

        # Patch the gaps (fixes the 3000 kPa vacuum)
        T_bubble = fill_missing_data(T_bubble)
        Hfg_mix = fill_missing_data(Hfg_mix)
        Cp_mix_liq = fill_missing_data(Cp_mix_liq)
        Cp_mix_vap = fill_missing_data(Cp_mix_vap)

        # --- Print formatted C Arrays ---
        print(f"// --- {P_kPa} kPa Operating Pressure ---")

        def print_c_array(name, data):
            array_str = ", ".join([f"{val:.4f}" for val in data])
            print(f"double {name}_{P_kPa}[101] = {{{array_str}}};")

        print_c_array("T_sat", T_bubble)
        print_c_array("H_fg", Hfg_mix)
        print_c_array("Cp_liq", Cp_mix_liq)
        print_c_array("Cp_vap", Cp_mix_vap)
        print("\n")

if __name__ == "__main__":
    generate_lookup_tables()

/* =========================================================================
   ANSYS Fluent UDF 1D Lookup Tables: Propane & Ammonia
   Y_propane resolution: 0.01 (101 points total)
========================================================================== */

// --- 971.5 kPa Operating Pressure ---
double T_sat_971.5[101] = {297.1283, 294.3662, 291.5623, 288.7292, 285.8820, 283.0384, 280.2194, 277.4494, 274.7562, 272.1701, 269.7225, 267.4438, 265.3612, 263.4965, 261.8644, 260.4719, 259.3180, 258.3948, 257.6885, 257.1817, 256.8540, 256.6844, 256.6518, 256.8290, 257.0062, 257.1835, 257.5153, 257.9016, 258.3313, 258.7950, 259.2848, 259.7937, 260.3160, 260.8467, 261.3817, 261.9173, 262.4506, 262.9789, 263.5000, 264.0122, 264.5136, 265.0031, 265.4794, 265.9415, 266.3885, 266.8199, 267.2348, 267.6329, 268.0138, 268.3769, 268.7222, 269.0401, 269.3580, 269.6483, 269.9201, 270.1733, 270.3894, 270.6055, 270.8216, 271.0008, 271.1618, 271.3048, 271.4300, 271.5290, 271.6280, 271.7015, 271.7585, 27

In [3]:
import CoolProp.CoolProp as CP

def get_saturation_properties(pressure_pa):
    """
    Calculates properties for BOTH saturated liquid and saturated vapor
    given the pressure in Pascals, including molar enthalpy (J/kmol) and Cp (J/kg/K).
    """
    fluid = 'Propane'

    try:
        print(f"==========================================================")
        print(f" SATURATION PROPERTIES FOR PROPANE AT {pressure_pa} Pa")
        print(f"==========================================================\n")

        # --- SATURATED LIQUID (Q = 0) ---
        d_liq = CP.PropsSI('D', 'P', pressure_pa, 'Q', 0, fluid)
        k_liq = CP.PropsSI('L', 'P', pressure_pa, 'Q', 0, fluid)
        v_liq = CP.PropsSI('V', 'P', pressure_pa, 'Q', 0, fluid)
        cp_liq = CP.PropsSI('C', 'P', pressure_pa, 'Q', 0, fluid) # Specific Heat (J/kg/K)
        # Hmolar is in J/mol. Multiply by 1000 to get J/kmol
        h_molar_liq = CP.PropsSI('Hmolar', 'P', pressure_pa, 'Q', 0, fluid) * 1000

        print("--- LIQUID PHASE (Q = 0) ---")
        print(f"Density:              {d_liq:.4f} kg/m^3")
        print(f"Specific Heat (Cp):   {cp_liq:.4f} J/kg/K")
        print(f"Thermal Conductivity: {k_liq:.6f} W/m/K")
        print(f"Viscosity:            {v_liq:.6e} Pa.s")
        print(f"Molar Enthalpy:       {h_molar_liq:.4f} J/kmol\n")

        # --- SATURATED VAPOR (Q = 1) ---
        d_vap = CP.PropsSI('D', 'P', pressure_pa, 'Q', 1, fluid)
        k_vap = CP.PropsSI('L', 'P', pressure_pa, 'Q', 1, fluid)
        v_vap = CP.PropsSI('V', 'P', pressure_pa, 'Q', 1, fluid)
        cp_vap = CP.PropsSI('C', 'P', pressure_pa, 'Q', 1, fluid) # Specific Heat (J/kg/K)
        # Hmolar is in J/mol. Multiply by 1000 to get J/kmol
        h_molar_vap = CP.PropsSI('Hmolar', 'P', pressure_pa, 'Q', 1, fluid) * 1000

        print("--- VAPOR PHASE (Q = 1) ---")
        print(f"Density:              {d_vap:.4f} kg/m^3")
        print(f"Specific Heat (Cp):   {cp_vap:.4f} J/kg/K")
        print(f"Thermal Conductivity: {k_vap:.6f} W/m/K")
        print(f"Viscosity:            {v_vap:.6e} Pa.s")
        print(f"Molar Enthalpy:       {h_molar_vap:.4f} J/kmol\n")

        # Quick Molar Latent Heat Calculation
        print(f"-> Calculated Molar Latent Heat: {(h_molar_vap - h_molar_liq):.4f} J/kmol\n")

    except ValueError as e:
        print(f"Error: {e}. Check if pressure is within valid saturation bounds.")


def get_superheated_vapor_properties(pressure_pa, temperature_k):
    """
    Calculates properties for a SUPERHEATED gas (above boiling point)
    given pressure in Pascals and temperature in Kelvin.
    """
    fluid = 'Propane'

    try:
        # Requires two independent inputs: Pressure ('P') and Temperature ('T')
        d_gas = CP.PropsSI('D', 'P', pressure_pa, 'T', temperature_k, fluid)
        k_gas = CP.PropsSI('L', 'P', pressure_pa, 'T', temperature_k, fluid)
        v_gas = CP.PropsSI('V', 'P', pressure_pa, 'T', temperature_k, fluid)
        cp_gas = CP.PropsSI('C', 'P', pressure_pa, 'T', temperature_k, fluid) # Specific Heat (J/kg/K)
        # Hmolar is in J/mol. Multiply by 1000 to get J/kmol
        h_molar_gas = CP.PropsSI('Hmolar', 'P', pressure_pa, 'T', temperature_k, fluid) * 1000

        print(f"--- SUPERHEATED VAPOR at {pressure_pa} Pa and {temperature_k} K ---")
        print(f"Density:              {d_gas:.4f} kg/m^3")
        print(f"Specific Heat (Cp):   {cp_gas:.4f} J/kg/K")
        print(f"Thermal Conductivity: {k_gas:.6f} W/m/K")
        print(f"Viscosity:            {v_gas:.6e} Pa.s")
        print(f"Molar Enthalpy:       {h_molar_gas:.4f} J/kmol\n")

    except ValueError as e:
        print(f"Error: {e}. Check if your inputs define a valid gas state.")

# ==========================================
# Example Execution
# ==========================================

# 1. Properties directly on the boiling curve (Liquid and Vapor limits)
input_pressure_pa = 971500  # 1 atm
get_saturation_properties(input_pressure_pa)

# 2. Properties for a pure, heated gas (Superheated Vapor)
input_pressure_gas = 101325
input_temperature_gas = 298.15
get_superheated_vapor_properties(input_pressure_gas, input_temperature_gas)

 SATURATION PROPERTIES FOR PROPANE AT 971500 Pa

--- LIQUID PHASE (Q = 0) ---
Density:              491.1136 kg/m^3
Specific Heat (Cp):   2727.6993 J/kg/K
Thermal Conductivity: 0.093492 W/m/K
Viscosity:            9.633147e-05 Pa.s
Molar Enthalpy:       11785554.9985 J/kmol

--- VAPOR PHASE (Q = 1) ---
Density:              21.0479 kg/m^3
Specific Heat (Cp):   2025.7954 J/kg/K
Thermal Conductivity: 0.019079 W/m/K
Viscosity:            8.299800e-06 Pa.s
Molar Enthalpy:       26528046.0401 J/kmol

-> Calculated Molar Latent Heat: 14742491.0416 J/kmol

--- SUPERHEATED VAPOR at 101325 Pa and 298.15 K ---
Density:              1.8320 kg/m^3
Specific Heat (Cp):   1684.6979 J/kg/K
Thermal Conductivity: 0.018310 W/m/K
Viscosity:            8.146086e-06 Pa.s
Molar Enthalpy:       27795005.1598 J/kmol



In [5]:
import CoolProp.CoolProp as CP

def calculate_blend_flow_rates(solute_name, P_kPa, Y_solute, Q_total_m3s=6.47707e-06):
    """
    Calculates the required mass flow rates of solute and solvent (Ammonia)
    to achieve a target volumetric flow rate at a given pressure.

    Parameters:
    -----------
    solute_name : str  - Name of the solute fluid (CoolProp string)
    P_kPa       : float - Operating pressure in kPa
    Y_solute    : float - Mass fraction of the solute (0.0 to 1.0)
    Q_total_m3s : float - Target total volumetric flow rate in m^3/s
    """

    solvent_name = "Ammonia"
    P_Pa = P_kPa * 1000.0
    Y_solvent = 1.0 - Y_solute

    try:
        # 1. Get Saturated Liquid Densities (kg/m^3) for pure components at input Pressure
        # 'D' = Density, 'P' = Pressure, 'Q' = Vapor Quality (0 = Sat Liquid)
        rho_solute = CP.PropsSI('D', 'P', P_Pa, 'Q', 0, solute_name)
        rho_solvent = CP.PropsSI('D', 'P', P_Pa, 'Q', 0, solvent_name)

        # 2. Calculate Total Mass Flow Rate (kg/s)
        # Using the ideal volume mixing rule: Q_tot = m_tot * (Y1/rho1 + Y2/rho2)
        specific_volume_mix = (Y_solute / rho_solute) + (Y_solvent / rho_solvent)
        m_dot_total = Q_total_m3s / specific_volume_mix

        # 3. Calculate Individual Mass Flow Rates (kg/s)
        m_dot_solute = m_dot_total * Y_solute
        m_dot_solvent = m_dot_total * Y_solvent

        # 4. Calculate Individual Volumetric Flow Rates (m^3/s) for verification
        Q_solute = m_dot_solute / rho_solute
        Q_solvent = m_dot_solvent / rho_solvent

        # --- Print Results ---
        print("==========================================================")
        print(f" Flow Rate Calculator: {solute_name} & {solvent_name}")
        print("==========================================================")
        print(f" Inputs:")
        print(f"   Pressure                = {P_kPa} kPa")
        print(f"   Target Volume Flow      = {Q_total_m3s:.5e} m^3/s")
        print(f"   Mass Fraction {solute_name:<9} = {Y_solute:.4f}")
        print(f"   Mass Fraction {solvent_name:<9} = {Y_solvent:.4f}\n")

        print(f" Saturated Liquid Densities:")
        print(f"   rho ({solute_name:<9})         = {rho_solute:.2f} kg/m^3")
        print(f"   rho ({solvent_name:<9})         = {rho_solvent:.2f} kg/m^3\n")

        print(f" Required MASS Flow Rates:")
        print(f"   m_dot ({solute_name:<9})       = {m_dot_solute:.5e} kg/s")
        print(f"   m_dot ({solvent_name:<9})       = {m_dot_solvent:.5e} kg/s")
        print(f"   ---------------------------------------")
        print(f"   m_dot (TOTAL)           = {m_dot_total:.5e} kg/s\n")

        print(f" Verification (Volumetric Flow Rates):")
        print(f"   Q ({solute_name:<9})           = {Q_solute:.5e} m^3/s")
        print(f"   Q ({solvent_name:<9})           = {Q_solvent:.5e} m^3/s")
        print(f"   ---------------------------------------")
        print(f"   Q (TOTAL)               = {(Q_solute + Q_solvent):.5e} m^3/s")
        print("==========================================================")

        return m_dot_solute, m_dot_solvent

    except ValueError as e:
        print(f"Error evaluating properties at {P_kPa} kPa. CoolProp says: {e}")
        print("Ensure the pressure is below the critical pressure of both pure fluids.")
        return None, None

# ==========================================
# Execute the Calculator
# ==========================================
if __name__ == "__main__":
    # Change these three variables to get desired output
    INPUT_SOLUTE = "Propane"
    INPUT_PRESSURE_KPA = 1971.5
    INPUT_MASS_FRACTION = 0.1  # 35% Solute, 65% Ammonia

    calculate_blend_flow_rates(
        solute_name=INPUT_SOLUTE,
        P_kPa=INPUT_PRESSURE_KPA,
        Y_solute=INPUT_MASS_FRACTION
    )

 Flow Rate Calculator: Propane & Ammonia
 Inputs:
   Pressure                = 1971.5 kPa
   Target Volume Flow      = 6.47707e-06 m^3/s
   Mass Fraction Propane   = 0.1000
   Mass Fraction Ammonia   = 0.9000

 Saturated Liquid Densities:
   rho (Propane  )         = 435.44 kg/m^3
   rho (Ammonia  )         = 565.00 kg/m^3

 Required MASS Flow Rates:
   m_dot (Propane  )       = 3.55379e-04 kg/s
   m_dot (Ammonia  )       = 3.19841e-03 kg/s
   ---------------------------------------
   m_dot (TOTAL)           = 3.55379e-03 kg/s

 Verification (Volumetric Flow Rates):
   Q (Propane  )           = 8.16133e-07 m^3/s
   Q (Ammonia  )           = 5.66094e-06 m^3/s
   ---------------------------------------
   Q (TOTAL)               = 6.47707e-06 m^3/s
